# 03 · PIB

Riqueza, produtividade e estrutura setorial.

**Fonte:** agregado 5938 (PIB municipal a preços correntes).

In [8]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import pandas as pd
from ibge_analytics.utils import io
from ibge_analytics.viz import charts, maps
from ibge_analytics.viz.theme import formatar_compacto, formatar_numero

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [9]:
from ibge_analytics.analysis import pib as an_pib

painel = io.carregar("painel_municipios")
painel_uf = io.carregar("painel_ufs")
pib_ufs = io.carregar("pib_ufs")
ano_pib = int(painel["ano_pib"].iloc[0])
ano_pop = int(painel["ano_populacao_pib"].iloc[0])
print(f"PIB {ano_pib} · população de referência {ano_pop}")
print(f"PIB nacional: R$ {painel['pib_mil_reais'].sum() * 1_000:,.0f}")

PIB 2023 · população de referência 2024
PIB nacional: R$ 10,943,345,420,000


> O PIB per capita usa a população do ano publicado mais próximo ao do PIB — a série de estimativas não cobre 2022 nem 2023.

## Concentração do PIB

In [10]:
conc = an_pib.resumo_concentracao_municipal(painel)
print(f"Metade do PIB nacional está em {conc['n_municipios_metade_pib']} municípios ({conc['pct_municipios_metade_pib']:.1f}% do total)")
print(f"Os 10 maiores somam {conc['share_top_10']:.1f}% do PIB")
print(f"Gini do PIB municipal: {conc['gini']:.3f}")

Metade do PIB nacional está em 84 municípios (1.5% do total)
Os 10 maiores somam 24.5% do PIB
Gini do PIB municipal: 0.839


## Riqueza x população por UF

In [11]:
desc = an_pib.descolamento_pib_populacao(painel_uf)
display(desc)
charts.barras_agrupadas_comparacao(
    desc, categoria="uf_sigla",
    series={"part_pib_brasil": "% do PIB", "part_pop_brasil": "% da população"},
    titulo="Onde a barra do PIB supera a da população, a UF concentra riqueza")

,uf_sigla,uf_nome,regiao_nome,part_pib_brasil,part_pop_brasil,razao_pib_pop,pib_per_capita
0,DF,Distrito Federal,Centro-Oeste,3.34,1.40,2.38,"122,591.83"
1,SP,São Paulo,Sudeste,31.48,21.59,1.46,"74,930.93"
2,MT,Mato Grosso,Centro-Oeste,2.49,1.82,1.37,"71,162.72"
3,RJ,Rio de Janeiro,Sudeste,10.72,8.07,1.33,"68,112.27"
4,MS,Mato Grosso do Sul,Centro-Oeste,1.69,1.37,1.23,"63,545.41"
5,SC,Santa Catarina,Sul,4.69,3.84,1.22,"63,708.72"
6,RS,Rio Grande do Sul,Sul,5.94,5.26,1.13,"57,890.64"
7,PR,Paraná,Sul,6.13,5.57,1.10,"56,738.96"
8,ES,Espírito Santo,Sudeste,1.92,1.93,0.99,"51,151.42"
9,MG,Minas Gerais,Sudeste,8.88,10.02,0.89,"45,584.19"


As duas séries são percentuais do mesmo total nacional, então compartilham um eixo legitimamente. Grandezas de escalas diferentes nunca deveriam dividir um eixo.

## Estrutura setorial

In [12]:
estrutura = an_pib.estrutura_setorial(painel_uf, chave="uf_sigla")
charts.barras_empilhadas(estrutura, x="uf_sigla", y="participacao", cor="setor",
                         titulo="Participação setorial no valor adicionado (%)")

## A economia está desconcentrando?

In [13]:
evolucao = an_pib.evolucao_participacao_regional(pib_ufs)
charts.linha_temporal(evolucao, x="ano", y="part_pib_brasil", cor="regiao_nome",
                      titulo="Participação de cada região no PIB nacional (%)")

## Riqueza x tamanho dos municípios

In [14]:
charts.dispersao_facetada(
    painel.dropna(subset=["pib_per_capita", "populacao_atual"]),
    x="populacao_atual", y="pib_per_capita",
    titulo="População (log) × PIB per capita (log), por região")

> Facetado em vez de colorido: numa dispersão todos os pares de cor competem entre si, e cinco séries sobrepostas deixariam de ser distinguíveis com segurança. A nuvem cinza é o país inteiro.